# Plotly : graphiques interactifs et 3D

Ce TP trace avec Plotly des graphiques qu'on explore à la souris : un nuage de points interactif, deux graphiques en trois dimensions, puis l'export d'un graphique en page web. Les exercices se font dans l'ordre que vous voulez ; le dernier est facultatif.

Les données sont fournies par Plotly lui-même (`plotly.express.data`) : rien à télécharger.

## Imports

In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## Les données

Le jeu principal vient de la fondation [Gapminder](https://www.gapminder.org/data/) : pour 142 pays, tous les cinq ans de 1952 à 2007, l'espérance de vie à la naissance (`lifeExp`, en années), la population (`pop`) et le PIB par habitant (`gdpPercap`, en dollars constants), avec le continent du pays (`continent`) et ses codes ISO (`iso_alpha`, `iso_num`). Une ligne est un pays une année donnée : 142 × 12 = 1 704 lignes. `pays_2007` n'en garde que l'année 2007, une ligne par pays.

In [ ]:
gapminder = px.data.gapminder()
pays_2007 = gapminder.query("year == 2007")
print(gapminder.shape, pays_2007.shape)
pays_2007.head()

## 1. Un nuage de points interactif

Tracez avec `px.scatter` l'espérance de vie des pays en 2007 en fonction de leur PIB par habitant : une couleur par continent, une taille de point qui suit la population, le nom du pays au survol, et le PIB en échelle logarithmique. Donnez aux axes et à la légende des titres en français (argument `labels`), et un titre à la figure. Appelez-la `fig_pib` : l'exercice 3 l'exporte.

Explorez ensuite la figure à la souris :

- en survolant les points, trouvez le pays d'Afrique dont l'espérance de vie est la plus faible, et le pays d'Asie dont le PIB par habitant est le plus élevé ;
- cliquez sur « Africa » dans la légende, puis double-cliquez dessus : que fait chacun de ces deux gestes ?
- retracez la figure sans l'échelle logarithmique : pourquoi est-elle préférable ici ?

**Documentation utile**

- [`plotly.express.scatter`](https://plotly.com/python-api-reference/generated/plotly.express.scatter.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`color`, `size` et `hover_name` reçoivent des noms de colonnes ; `log_x=True` passe l'axe des abscisses en échelle logarithmique. `labels` est un dictionnaire qui associe à un nom de colonne le titre à afficher (`"pop"` sert à la bulle du survol), `title` donne le titre de la figure. Le pays d'Afrique cherché a une espérance de vie inférieure à 40 ans.

```python
fig_pib = px.scatter(
  pays_2007,
  x="gdpPercap",
  y="lifeExp",
  color=...,
  size=...,
  hover_name=...,
  log_x=...,
  labels={"gdpPercap": ..., "lifeExp": ..., "pop": ..., "continent": ...},
  title=...,
)
fig_pib.show()
```

### Solution

In [ ]:
fig_pib = px.scatter(
  pays_2007,
  x="gdpPercap",
  y="lifeExp",
  color="continent",
  size="pop",
  hover_name="country",
  log_x=True,
  labels={
    "gdpPercap": "PIB par habitant (dollars)",
    "lifeExp": "Espérance de vie (années)",
    "pop": "Population",
    "continent": "Continent",
  },
  title="Espérance de vie et richesse des pays en 2007",
)
fig_pib.show()
# Afrique : l'Eswatini (Swaziland dans les données), 39,6 ans.
# Asie : le Koweït, 47 307 dollars par habitant.
# Un clic sur un continent de la légende le masque ; un double-clic n'affiche que lui.
# Sans échelle logarithmique, les trois quarts des pays (moins de 20 000 dollars)
# s'entassent à gauche : le log montre des rapports (x 10) plutôt que des écarts.

## 2. La 3D

Plotly dessine aussi en trois dimensions : la figure tourne quand on la fait glisser à la souris, et la molette zoome.

### 2.1 Les iris en trois dimensions

Les iris de Fisher (`px.data.iris()`) décrivent 150 fleurs de trois espèces (`species`) par quatre mesures en centimètres : la longueur et la largeur du sépale (`sepal_length`, `sepal_width`) et du pétale (`petal_length`, `petal_width`).

Tracez avec `px.scatter_3d` le nuage de points de trois de ces mesures, une couleur par espèce, puis faites tourner la figure :

- quelle espèce se sépare des deux autres sous tous les angles ?
- trouvez un angle où *versicolor* et *virginica* se recouvrent, et un autre où elles se séparent presque : que dit cette différence de ce qu'une seule vue en deux dimensions peut cacher ?

In [ ]:
iris = px.data.iris()

# Votre code ici

#### Conseils supplémentaires

Les arguments `x`, `y` et `z` reçoivent chacun un nom de colonne, `color` celui de la colonne de l'espèce. Les mesures des pétales séparent le mieux les espèces : prenez-en au moins une.

```python
fig_iris = px.scatter_3d(iris, x=..., y=..., z=..., color=...)
fig_iris.show()
```

#### Solution

In [ ]:
fig_iris = px.scatter_3d(
  iris, x="sepal_length", y="petal_length", z="petal_width", color="species"
)
fig_iris.show()
# setosa, aux petits pétales, reste à l'écart sous tous les angles.
# Vues dans l'axe des pétales, versicolor et virginica se recouvrent ; vues de face
# sur la longueur et la largeur des pétales, elles se séparent presque. Une projection
# en deux dimensions peut cacher une séparation que la rotation fait apparaître.

### 2.2 Une surface : le point selle

La fonction $f(x, y) = x^2 - y^2$ a un *point selle* en $(0, 0)$ : sa pente y est nulle dans toutes les directions, comme en un minimum, et pourtant ce n'en est pas un. Les algorithmes d'optimisation, comme la descente de gradient, en rencontrent souvent.

Calculez $f$ sur une grille de 81 × 81 points du carré $[-2, 2] \times [-2, 2]$ (`np.linspace`, `np.meshgrid`), puis tracez la surface avec `go.Surface`. Ajoutez les lignes de niveau, projetées sous la surface (`contours_z`). Faites tourner la figure : dans quelle direction $(0, 0)$ est-il un minimum, dans laquelle un maximum ?

**Documentation utile**

- [Surfaces 3D avec Plotly](https://plotly.com/python/3d-surface-plots/)

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`np.meshgrid(x, y)` renvoie deux tableaux de 81 × 81 valeurs, l'abscisse et l'ordonnée de chaque point de la grille : $f$ se calcule sur eux en une seule expression, sans boucle. `go.Surface` reçoit `x` et `y` à une dimension, et `z` à deux. `update_traces(contours_z={"show": True, "project_z": True})` dessine les lignes de niveau de `z` et les projette sur le plan du bas.

```python
x = np.linspace(-2, 2, 81)
y = ...
X, Y = np.meshgrid(x, y)
Z = ...
fig_selle = go.Figure(go.Surface(x=x, y=y, z=Z))
fig_selle.update_traces(contours_z=...)
fig_selle.show()
```

#### Solution

In [ ]:
x = np.linspace(-2, 2, 81)
y = np.linspace(-2, 2, 81)
X, Y = np.meshgrid(x, y)
Z = X**2 - Y**2
fig_selle = go.Figure(go.Surface(x=x, y=y, z=Z))
fig_selle.update_traces(contours_z={"show": True, "project_z": True})
fig_selle.update_layout(title="f(x, y) = x² − y²")
fig_selle.show()
# Le long de l'axe des x (y = 0), f vaut x² : (0, 0) y est un minimum.
# Le long de l'axe des y (x = 0), f vaut −y² : (0, 0) y est un maximum.

## 3. Exporter une page web

Un graphique Plotly s'enregistre en une page HTML qui s'ouvre dans n'importe quel navigateur, interactivité comprise.

Enregistrez `fig_pib` (exercice 1) avec `write_html` deux fois : dans `pib.html` avec les réglages par défaut, puis dans `pib_cdn.html` avec `include_plotlyjs="cdn"`. Affichez la taille des deux fichiers, en kilo-octets. Ouvrez-les dans le navigateur (dans Colab : le panneau *Fichiers*, à gauche, puis *Télécharger* ; dans un Codespace : l'explorateur de fichiers) : d'où vient la différence de taille, et lequel enverriez-vous à quelqu'un qui le lira sans connexion à Internet ?

L'export en image statique (`write_image`, en PNG, SVG ou PDF) demande en plus le paquet `kaleido` et un navigateur Chrome, que Colab n'installe pas : ce TP s'en tient au HTML.

**Documentation utile**

- [`pathlib.Path.stat`](https://docs.python.org/3/library/pathlib.html#pathlib.Path.stat)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`fig.write_html(chemin)` écrit la page. Son argument `include_plotlyjs` dit comment elle obtient plotly.js, la bibliothèque JavaScript qui dessine le graphique : copiée dans la page (`True`, par défaut) ou téléchargée à l'ouverture depuis un CDN, un serveur public (`"cdn"`). `Path(chemin).stat().st_size` donne la taille d'un fichier en octets. La page par défaut pèse plusieurs mégaoctets.

```python
fig_pib.write_html(...)
fig_pib.write_html(..., include_plotlyjs=...)
for chemin in ["pib.html", "pib_cdn.html"]:
  print(chemin, ..., "Ko")
```

### Solution

In [ ]:
fig_pib.write_html("pib.html")
fig_pib.write_html("pib_cdn.html", include_plotlyjs="cdn")
for chemin in ["pib.html", "pib_cdn.html"]:
  print(chemin, Path(chemin).stat().st_size // 1000, "Ko")
# pib.html contient plotly.js (plus de 4 Mo) : il s'ouvre sans connexion.
# pib_cdn.html (une quinzaine de Ko) ne contient que les données et télécharge
# plotly.js à l'ouverture : bien plus léger, mais il lui faut Internet.

## 4. Facultatif : animer les années 1952 à 2007

Retracez le nuage de l'exercice 1 sur toutes les années de `gapminder`, animé année par année : `animation_frame` dit quelle colonne fait défiler les images, `animation_group` quelle colonne relie un point d'une image au point de l'image suivante (ici, le pays). Fixez les bornes des axes avec `range_x` et `range_y`, pour qu'elles contiennent toutes les années.

Lancez l'animation avec le bouton ▶, ou faites glisser le curseur :

- comment la Chine évolue-t-elle de 1952 à 2007 ?
- un point s'effondre en 1992 : lequel, et pourquoi ?
- retirez `range_x` et `range_y` : que se passe-t-il ?

**Documentation utile**

- [Animations avec Plotly Express](https://plotly.com/python/animations/)

In [ ]:
# Votre code ici

### Conseils supplémentaires

De 1952 à 2007, le PIB par habitant va d'environ 240 à 114 000 dollars, et l'espérance de vie de 24 à 83 ans. Avec `log_x=True`, `range_x` s'écrit quand même en dollars : `[100, 150_000]` couvre tout.

```python
fig_animee = px.scatter(
  gapminder,
  x="gdpPercap",
  y="lifeExp",
  color="continent",
  size="pop",
  hover_name="country",
  log_x=True,
  animation_frame=...,
  animation_group=...,
  range_x=[..., ...],
  range_y=[..., ...],
)
fig_animee.show()
```

### Solution

In [ ]:
fig_animee = px.scatter(
  gapminder,
  x="gdpPercap",
  y="lifeExp",
  color="continent",
  size="pop",
  hover_name="country",
  log_x=True,
  animation_frame="year",
  animation_group="country",
  range_x=[100, 150_000],
  range_y=[20, 90],
  labels={
    "gdpPercap": "PIB par habitant (dollars)",
    "lifeExp": "Espérance de vie (années)",
    "year": "Année",
  },
)
fig_animee.show()
# La Chine passe de 44 à 73 ans d'espérance de vie, et de 400 à près de
# 5 000 dollars par habitant.
# Le Rwanda tombe à 23,6 ans en 1992 : la guerre civile commencée en 1990, puis le
# génocide des Tutsi de 1994.
# Sans range_x ni range_y, Plotly Express ne calcule pas de bornes valables pour toutes
# les années : des points sortent du cadre au fil de l'animation, et l'échelle peut
# changer d'une image à l'autre, ce qui fausse la comparaison.